# B6 - U-Net Noise Predictor (DDPM forward process) on Oxford-IIIT Pet
- Images resized to 64x64, scaled to [-1, 1]
- Linear noise schedule, T = 200: $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon$
- U-Net (as in the table) predicts $\epsilon$; MSE loss

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

In [ ]:
# 1. Load images and resize to 64x64
IMG, BATCH, T = 64, 32, 200

def preprocess(sample):
    img = tf.image.resize(sample['image'], (IMG, IMG))
    return tf.cast(img, tf.float32) / 127.5 - 1.0      # [-1, 1]

train_raw = tfds.load('oxford_iiit_pet', split='train')
test_raw = tfds.load('oxford_iiit_pet', split='test')
train_images = train_raw.map(preprocess)
test_images = test_raw.map(preprocess)

In [ ]:
# 2. Linear noise schedule (T = 200) + forward diffusion
# betas scaled by 1000/T so the image is almost pure noise at t = T
betas = np.linspace(1e-4 * 1000 / T, 0.02 * 1000 / T, T).astype("float32")
alpha_bar = np.concatenate([[1.0], np.cumprod(1.0 - betas)]).astype("float32")   # index t = 0..T (t=0 is clean)
alpha_bar_tf = tf.constant(alpha_bar)

def add_noise(x0, t):
    ab = tf.reshape(tf.gather(alpha_bar_tf, t), (-1, 1, 1, 1))
    eps = tf.random.normal(tf.shape(x0))
    return tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * eps, eps

def make_pair(x0):                       # random t per image
    t = tf.random.uniform((tf.shape(x0)[0],), 1, T + 1, dtype=tf.int32)
    xt, eps = add_noise(x0, t)
    return xt, eps

train_ds = train_images.shuffle(1000).batch(BATCH, drop_remainder=True).map(make_pair).prefetch(tf.data.AUTOTUNE)
test_ds = test_images.batch(BATCH).map(make_pair)

In [ ]:
# 3. Noisy examples at t = 0, 50, 100, 150, 200
x0 = next(iter(train_images.batch(1)))
steps = [0, 50, 100, 150, 200]
plt.figure(figsize=(12, 3))
for i, t in enumerate(steps):
    xt, _ = add_noise(x0, tf.constant([t]))
    plt.subplot(1, 5, i + 1); plt.imshow(np.clip((xt[0] + 1) / 2, 0, 1)); plt.title(f"t={t}"); plt.axis('off')
plt.show()

In [ ]:
# U-Net
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    return x

def build_unet():
    inp = layers.Input((IMG, IMG, 3))
    e1 = conv_block(inp, 32)                         # 64x64x32 (skip)
    p1 = layers.MaxPooling2D(2)(e1)                  # 32x32x32
    e2 = conv_block(p1, 64)                          # 32x32x64 (skip)
    p2 = layers.MaxPooling2D(2)(e2)                  # 16x16x64
    b = conv_block(p2, 128)                          # 16x16x128
    u1 = layers.UpSampling2D(2)(b)                   # 32x32x128
    u1 = layers.Concatenate()([u1, e2])              # 32x32x192
    d1 = conv_block(u1, 64)                          # 32x32x64
    u2 = layers.UpSampling2D(2)(d1)                  # 64x64x64
    u2 = layers.Concatenate()([u2, e1])              # 64x64x96
    d2 = conv_block(u2, 32)                          # 64x64x32
    out = layers.Conv2D(3, 1, activation='linear')(d2)   # predicted noise
    return keras.Model(inp, out, name='unet')

unet = build_unet()
unet.summary()

In [ ]:
# 4. Train with MSE to predict the added noise
unet.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
history = unet.fit(train_ds, epochs=5)
print("Test MSE:", unet.evaluate(test_ds, verbose=0))

In [ ]:
# 5. Training loss + actual vs predicted noise
plt.plot(history.history['loss']); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Training loss'); plt.show()

x0 = next(iter(test_images.batch(3)))
t = tf.constant([50, 100, 150])
xt, eps = add_noise(x0, t)
eps_pred = unet.predict(xt, verbose=0)

show = lambda a: np.clip((a + 2) / 4, 0, 1)      # map noise ~[-2,2] to [0,1] for display
plt.figure(figsize=(9, 9))
for i in range(3):
    plt.subplot(3, 3, 3 * i + 1); plt.imshow(np.clip((xt[i] + 1) / 2, 0, 1)); plt.title(f'noisy t={int(t[i])}'); plt.axis('off')
    plt.subplot(3, 3, 3 * i + 2); plt.imshow(show(eps[i].numpy())); plt.title('actual noise'); plt.axis('off')
    plt.subplot(3, 3, 3 * i + 3); plt.imshow(show(eps_pred[i])); plt.title('predicted noise'); plt.axis('off')
plt.show()